In [40]:
!pip install requests

In [41]:
import json
import requests

# Base API URL for Chicago Crime dataset
base_url = "https://data.cityofchicago.org/resource/ijzp-q8t2.json"

# Define the target date ranges for Q1 2025 month-by-month
months = [
    {"name": "January", "start": "2025-01-01T00:00:00", "end": "2025-01-31T23:59:59"},
    {"name": "February", "start": "2025-02-01T00:00:00", "end": "2025-02-28T23:59:59"},
    {"name": "March", "start": "2025-03-01T00:00:00", "end": "2025-03-31T23:59:59"}
]

# Initialize our master list
all_records = []

print("--- Starting Phase 1: Data Extraction ---")

# Loop through each month and request data from the API
for month in months:
    # FIXED: Using single quotes on the outside, double quotes on the inside for the date strings
    params = {
        "$where": f"date >= '{month['start']}' AND date <= '{month['end']}'",
        "$limit": 50000,    
        "$order": "date ASC" 
    }

    try:
        # Make the HTTP GET request
        response = requests.get(base_url, params=params)
        response.raise_for_status() 

        # Parse JSON list
        month_records = response.json()
        print(f"Successfully extracted {len(month_records)} records for {month['name']}.")

        # Append to master list
        all_records.extend(month_records)

    except requests.exceptions.RequestException as e:
        print(f"❌ Error fetching data for {month['name']}: {e}")

# Print total length of the combined records
total_count = len(all_records)
print(f"\n--- Extraction Complete ---")
print(f"Total combined records collected: {total_count}")

# Checkpoint Validation Logic
if 40000 <= total_count <= 60000:
    print("✅ Checkpoint Passed: Total row count is within the expected range!")
else:
    print("⚠️ Checkpoint Failed: The row count looks incorrect.")

--- Starting Phase 1: Data Extraction ---
Successfully extracted 18519 records for January.
Successfully extracted 16544 records for February.
Successfully extracted 19786 records for March.

--- Extraction Complete ---
Total combined records collected: 54849
✅ Checkpoint Passed: Total row count is within the expected range!


In [42]:
import pandas as pd

In [43]:
# Convert the raw list of dictionaries directly into a Pandas DataFrame
df = pd.DataFrame(all_records)

# Define a dictionary mapping the Raw API fields to our clean target names
column_mapping = {
    "id": "crime_id",               
    "case_number": "case_number",   
    "date": "occurred_on",         
    "block": "block",               
    "primary_type": "primary_type", 
    "description": "description",   
    "location_description": "location_description", 
    "arrest": "arrest",             
    "domestic": "domestic",         
    "beat": "beat",                 
    "district": "district",         
    "ward": "ward",                 
    "community_area": "community_area", 
    "fbi_code": "fbi_code",         
    "year": "year",                 
    "updated_on": "updated_on",     
    "latitude": "latitude",         
    "longitude": "longitude"        
}

# Safely filter for columns that actually exist in the API response to avoid KeyError
existing_columns = [col for col in column_mapping.keys() if col in df.columns]
df = df[existing_columns]

# Rename the columns to match our target database schema mapping layout
df = df.rename(columns=column_mapping)

print(f"DataFrame built successfully with shape: {df.shape}")

DataFrame built successfully with shape: (54849, 18)


In [44]:
# 1. Convert Date strings to proper datetime objects
df["occurred_on"] = pd.to_datetime(df["occurred_on"])
df["updated_on"] = pd.to_datetime(df["updated_on"])

# 2. Standardize text columns to UPPERCASE and strip out loose whitespaces
text_cols = ["primary_type", "description", "location_description", "block"]
for col in text_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.upper().str.strip()

print("Dates converted and text columns standardized successfully.")

Dates converted and text columns standardized successfully.


In [45]:

# 1. Convert boolean text strings to safe tiny integers (1 and 0)
# Force map string representations cleanly to integers 
df["arrest"] = df["arrest"].astype(str).str.lower().map({"true": 1, "false": 0})
df["domestic"] = df["domestic"].astype(str).str.lower().map({"true": 1, "false": 0})

# Explicitly replace any lingering NaN values with None
df["arrest"] = df["arrest"].where(pd.notnull(df["arrest"]), None)

# 2. Convert coordinates to floating decimals
df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")

# 3. Convert administrative codes to Pandas nullable integers ("Int64")
nullable_int_cols = ["ward", "community_area", "beat", "district"]
for col in nullable_int_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

# Convert year to regular int since it won't have any missing values
df["year"] = pd.to_numeric(df["year"], errors="coerce").astype(int)

print("Data type conversions successfully applied.")

Data type conversions successfully applied.


In [46]:
# 1. Fill missing descriptive text fields with fallback placeholders
df["location_description"] = df["location_description"].fillna("UNKNOWN")
df["description"] = df["description"].fillna("NOT SPECIFIED")

# 2. Check and eliminate duplicate records based on the unique crime_id
duplicate_count = df["crime_id"].duplicated().sum()
print(f"Found {duplicate_count} duplicate records.")

df = df.drop_duplicates(subset=["crime_id"], keep="first")
print(f"Duplicates removed. Current rows left: {df.shape[0]}")

Found 0 duplicate records.
Duplicates removed. Current rows left: 54849


In [47]:
# Define our lookup sets based on instructions
violent_codes = {"01A", "01B", "02", "03", "04A", "04B"}
property_codes = {"05", "06", "07", "08A", "08B", "09", "10", "11"}

def categorize_severity(fbi_code):
    if pd.isna(fbi_code):
        return "Other"
    
    code_str = str(fbi_code).strip().zfill(2).upper()
    
    if code_str in violent_codes:
        return "Violent"
    elif code_str in property_codes:
        return "Property"
    else:
        return "Other"

# Generate our brand new feature row
df["severity_tier"] = df["fbi_code"].apply(categorize_severity)

print("Severity tiers successfully calculated. Current breakdown counts:")
print(df["severity_tier"].value_counts())

Severity tiers successfully calculated. Current breakdown counts:
severity_tier
Property    34768
Other       15303
Violent      4778
Name: count, dtype: int64


In [48]:
import os

# Create the data folder if it doesn't exist yet
os.makedirs("data", exist_ok=True)

# Export to a flat CSV file tracking checkpoint status
output_file_path = "data/chicago_crimes_q1_2025_clean.csv"
df.to_csv(output_file_path, index=False)

print(f"✅ Checkpoint Passed: Cleaned CSV saved successfully to '{output_file_path}'!")
print(f"Final Data Matrix Cleaned Shape: {df.shape}")

✅ Checkpoint Passed: Cleaned CSV saved successfully to 'data/chicago_crimes_q1_2025_clean.csv'!
Final Data Matrix Cleaned Shape: (54849, 19)


In [49]:
  !pip install mysql-connector-python

In [50]:
import mysql.connector
import pandas as pd
import numpy as np

print("--- Starting Phase 4: Database Insertion Process ---")

# 1. Read Cleaned Checkpoint CSV from your data folder
csv_file = "data/chicago_crimes_q1_2025_clean.csv"
df = pd.read_csv(csv_file)

# 2. Fix schema datatypes that flat CSV parsing inherently breaks [cite: 300]
df["occurred_on"] = pd.to_datetime(df["occurred_on"])
df["updated_on"] = pd.to_datetime(df["updated_on"])

# Convert numeric columns explicitly, keeping them as numeric object types for now
numeric_cols = ["latitude", "longitude", "ward", "community_area", "beat", "district", "year"]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# 3. CRITICAL FIXED STEP: Convert DataFrame to native Python records to wipe out NumPy NaNs 
# We convert the selected columns into a list of dictionaries, replacing all float 'nan' with proper Python None
target_columns = [
    "crime_id", "case_number", "occurred_on", "block", "primary_type",
    "description", "location_description", "arrest", "domestic", "beat",
    "district", "ward", "community_area", "fbi_code", "year",
    "updated_on", "latitude", "longitude", "severity_tier"
]

raw_records = df[target_columns].to_dict(orient="records")

tuples_list = []
for record in raw_records:
    # Double check every single value in the record dictionary
    clean_row = tuple(
        None if (isinstance(val, float) and np.isnan(val)) or pd.isna(val) else val 
        for val in record.values()
    )
    tuples_list.append(clean_row)

# 4. Database Connection Configuration Profile
db_config = {
    "host": "localhost",
    "user": "root",  
    "password": "Dileep@6702",  # <--- Change this to your actual MySQL password again!
    "database": "chicago_crime_db"
}

try:
    # Connect to your local MySQL instance
    cnx = mysql.connector.connect(**db_config)
    cursor = cnx.cursor()
    print("Database connection successfully established.")

    # 5. Construct Idempotent Upsert Operation string [cite: 341]
    insert_query = """
        INSERT IGNORE INTO crimes (
            crime_id, case_number, occurred_on, block, primary_type,
            description, location_description, arrest, domestic, beat,
            district, ward, community_area, fbi_code, year,
            updated_on, latitude, longitude, severity_tier
        ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    """

    # 6. Execute bulk transaction block execution [cite: 356]
    print(f"Beginning bulk insert execution of {len(tuples_list)} rows...")
    cursor.executemany(insert_query, tuples_list) #

    # Commit changes safely to the database [cite: 357]
    cnx.commit()
    print(f"\n✅ Success! Inserted {cursor.rowcount} rows into MySQL database successfully.") #

except mysql.connector.Error as err:
    print(f"❌ Database operation failed critically: {err}")

finally:
    # Close resources safely [cite: 359, 360]
    if 'cursor' in locals():
        cursor.close()
    if 'cnx' in locals():
        cnx.close()
    print("Database connection pool references safely disconnected.")

--- Starting Phase 4: Database Insertion Process ---
Database connection successfully established.
Beginning bulk insert execution of 54849 rows...

✅ Success! Inserted 54849 rows into MySQL database successfully.
Database connection pool references safely disconnected.


In [1]:
import os

# Define the folder path we just created
project_folder = "C:\\Users\\balle\\chicago_crime_etl_pipeline"

# Check if the folder exists and list what is inside
if os.path.exists(project_folder):
    print("✅ Yes! Your folder exists.")
    print("📁 Files inside your project folder:")
    print(os.listdir(project_folder))
else:
    print("❌ The folder was not found. Let's look inside your main folder instead:")
    print(os.listdir("C:\\Users\\balle\\"))

✅ Yes! Your folder exists.
📁 Files inside your project folder:
['.git', '.gitignore', '.ipynb_checkpoints', 'chicago_crime_project.ipynb', 'output']


In [2]:
import os
import shutil

# Target project directory path
project_dir = "C:\\Users\\balle\\chicago_crime_etl_pipeline"

# Let's create proper folders for your SQL scripts and Power BI output charts
os.makedirs(os.path.join(project_dir, "sql"), exist_ok=True)
os.makedirs(os.path.join(project_dir, "output"), exist_ok=True)

# 1. Look for your SQL script in your main folder and move it safely
src_sql = "C:\\Users\\balle\\create_table.sql"
dst_sql = os.path.join(project_dir, "sql", "create_table.sql")
if os.path.exists(src_sql):
    shutil.move(src_sql, dst_sql)
    print("✅ Moved your SQL script into the project folder!")

# 2. Look for your Power BI dashboard file and move it safely
src_pbix = "C:\\Users\\balle\\chicago_crime_dashboard.pbix"
dst_pbix = os.path.join(project_dir, "chicago_crime_dashboard.pbix")
if os.path.exists(src_pbix):
    shutil.move(src_pbix, dst_pbix)
    print("✅ Moved your Power BI dashboard file into the project folder!")

print("\n🎉 All project elements are officially grouped together in your pipeline folder!")


🎉 All project elements are officially grouped together in your pipeline folder!
